# Lab 12 - PromptLayer-Style Instrumentation (Local)

**Week 2, Day 4.** You will add PromptLayer-style instrumentation to a small
LangChain-style pipeline **without any external calls**. You log every run
(prompt, variables, response, latency, approximate tokens), tag it with a
`prompt_version` and a `dataset_hash`, then filter, export, and compare two
prompt versions the way a PromptLayer dashboard would.

**Runtime.** The lab has three backends, selected with the `LAB12_BACKEND`
environment variable: `offline` (default), `lmstudio`, and `ollama`. Offline is a
deterministic stand-in so the lab runs anywhere and gives the same numbers every
time. `lmstudio` and `ollama` point at a local OpenAI-compatible server. Switching
backends is a one-flag change and nothing else in the notebook moves. That is the
point: the instrumentation is provider-agnostic.

## Learning goals

By the end of this lab you will be able to:

1. Add logging and tracing to a chain call: prompt, variables, response,
   latency, and approximate token count.
2. Record `prompt_version`, `dataset_hash`, and tags on every run.
3. Filter runs by tag and version, then export to CSV.
4. Compare two prompt versions on validity, token cost, and output drift.
5. Keep the code provider-agnostic so LM Studio or Ollama swaps in with one flag.

## How this lab is wired

Two design choices are worth calling out before you start.

**Offline by default.** The model is a deterministic stand-in whose output is a
fixed function of the prompt text. That makes the lab reproducible and gradeable
without a GPU or a network. Against LM Studio or Ollama you will see real, noisier
numbers, but the instrumentation code does not change. The stand-in is a teaching
device, not a real model.

**Sentinel prompts, not a template engine.** The few-shot prompt contains literal
JSON braces. A templating engine would try to read those braces as variables and
crash. We fill prompts with plain string replacement of `[[EMAIL]]` and
`[[LEAD_JSON]]` instead, so literal braces pass through untouched.

In [ ]:
# --- Config and working directories -------------------------------------
import os, re, csv, json, time, uuid, hashlib, statistics
from pathlib import Path
from typing import Any, Optional, Literal
from pydantic import BaseModel, ValidationError
from rapidfuzz import fuzz

# Backend selects the model runtime. Three modes, set with LAB12_BACKEND:
#   "offline"  deterministic stand-in, no network, fully reproducible (default)
#   "lmstudio" LM Studio OpenAI-compatible server on port 1234
#   "ollama"   Ollama OpenAI-compatible server on port 11434
# LM Studio and Ollama both speak the OpenAI API, so one client handles both.
# The rest of the lab does not change: the instrumentation is provider-agnostic.
BACKEND = os.getenv("LAB12_BACKEND", "offline")

# Per-backend connection defaults for the two local servers. Override any of
# these with LAB12_BASE_URL, LAB12_MODEL, LAB12_API_KEY if your setup differs.
LOCAL_SERVERS = {
    "lmstudio": {"base_url": "http://localhost:1234/v1", "model": "local-model"},
    "ollama":   {"base_url": "http://localhost:11434/v1", "model": "gemma4"},
}

ROOT = Path("lab12_workspace")
for d in ["datasets", "prompts", "logging_local/artifacts", "responses", "exports"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

RUNS_PATH = ROOT / "logging_local/artifacts/runs.jsonl"
SESSIONS_PATH = ROOT / "logging_local/artifacts/sessions.jsonl"

def reset_logs():
    "Start each full build from empty logs so run counts are reproducible."
    for p in (RUNS_PATH, SESSIONS_PATH):
        if p.exists():
            p.unlink()

reset_logs()
print("workspace:", ROOT.resolve())
print("backend:", BACKEND)

## Part A - Minimal dataset and prompts

Three synthetic sales emails and three prompts: a baseline extractor, a few-shot
extractor, and a summarizer. All synthetic, all fictional.

In [ ]:
# --- Part A: synthetic dataset ------------------------------------------
LEADS = {
    "records": [
        {"id": "S01", "email": "Body: I am Priya at CloudWave. Need EU data residency and SSO. Seats: 25."},
        {"id": "S02", "email": "Body: Hello from Pine and Co. We need pricing for 60 seats. Must support SAML."},
        {"id": "S03", "email": "Body: Hi, our trial ends next week. We want to convert to 10 editor and 40 viewer. Annual billing?"},
    ]
}
DATA_PATH = ROOT / "datasets/leads_mini.json"
DATA_PATH.write_text(json.dumps(LEADS, indent=2), encoding="utf-8")
print("wrote", DATA_PATH, "records:", len(LEADS["records"]))

In [ ]:
# --- Part A: prompts (sentinel placeholders, no template engine) --------
# We fill [[EMAIL]] and [[LEAD_JSON]] with plain string replacement. This
# avoids the footgun where a templating engine treats literal JSON braces in
# few-shot examples as variables and crashes.
EXTRACT_V1 = """ROLE: Sales Ops extractor
INSTRUCTION:
Return STRICT JSON with keys: company, need, requirements, seat_counts.
need must be one of: pricing, eu_residency, trial_to_paid, feature_question, other.
Use only facts present in the email. Do not invent values.
EMAIL: [[EMAIL]]
OUTPUT: JSON only. No prose. No code fences."""

EXTRACT_V2 = """ROLE: Sales Ops extractor
GUIDE EXAMPLES (few-shot):
- "Need EU data residency and SSO. Seats: 25." maps to need eu_residency, requirements SSO, viewer 25.
- "Pricing for 60 seats. Must support SAML." maps to need pricing, requirements SAML, viewer 60.
- "Convert to 10 editor and 40 viewer. Annual?" maps to need trial_to_paid, requirements annual_billing, editor 10 viewer 40.
INSTRUCTION:
Return STRICT JSON with keys: company, need, requirements, seat_counts.
need must be one of: pricing, eu_residency, trial_to_paid, feature_question, other.
Always capture the company when stated. Use facts only.
EMAIL: [[EMAIL]]
OUTPUT: JSON only. No prose. No code fences."""

SUMMARIZE = """ROLE: Executive summarizer
INSTRUCTION:
Given a validated lead JSON, produce a short Markdown brief with:
- Company and Need
- Requirements (up to 3)
- Seats if known
Keep it concise. No code fences.
LEAD_JSON: [[LEAD_JSON]]
OUTPUT: Markdown only."""

for name, text in {"v1": EXTRACT_V1, "v2": EXTRACT_V2, "summarize": SUMMARIZE}.items():
    (ROOT / f"prompts/{name}.txt").write_text(text, encoding="utf-8")
print("wrote 3 prompt files")

## Part B - Schema, parsing, and the logging shim

You will validate extracted leads with a pydantic v2 model, parse model text
robustly, and build the instrumentation shim that writes one JSON line per run.

In [ ]:
# --- Part B: lead schema (pydantic v2) ----------------------------------
# TODO 1. Define the Lead schema and validate_lead.
#
# Contract:
#   NeedT is one of: pricing, eu_residency, trial_to_paid, feature_question, other
#   Lead fields:
#     company: optional string, default None
#     need: NeedT, required, no default
#     requirements: list of strings, default empty list
#     seat_counts: object with optional integer editor and viewer, both default None
#   validate_lead(obj) returns [] when obj validates as a Lead, otherwise a
#     non-empty list of error strings. Do not raise on invalid input.

NeedT = None  # replace

class SeatCounts(BaseModel):
    pass  # replace

class Lead(BaseModel):
    pass  # replace

def validate_lead(obj: dict[str, Any]) -> list[str]:
    """Return [] when obj is a valid Lead, else a list of error strings."""
    raise NotImplementedError

print("schema stub loaded")

In [ ]:
# --- Part B: robust JSON parsing ----------------------------------------
# TODO 2. Implement as_json.
#
# Contract:
#   Input is raw model text that should be a JSON object. It may arrive wrapped
#   in a Markdown code fence. Return the parsed dict, or {} on any failure.
#   Never raise.
def as_json(text: str) -> dict[str, Any]:
    """Parse model text into a dict; return {} on failure."""
    raise NotImplementedError

print("as_json stub loaded")

In [ ]:
# --- Soft check helper --------------------------------------------------
_RESULTS = []
def check(name, thunk):
    try:
        status = "PASS" if bool(thunk()) else "FAIL"
    except NotImplementedError:
        status = "TODO"
    except Exception as e:
        status = "ERROR"
        print(f"    ({type(e).__name__}: {e})")
    _RESULTS.append(status)
    print(f"[{status}] {name}")
    return status == "PASS"

def check_summary():
    from collections import Counter
    c = Counter(_RESULTS)
    print(f"\n{c['PASS']} PASS / {c['FAIL']} FAIL / {c['TODO']} TODO / {c['ERROR']} ERROR / {len(_RESULTS)} total")

In [ ]:
# Checks for schema and parsing
check("as_json plain object", lambda: as_json('{"need": "pricing"}') == {"need": "pricing"})
check("as_json fenced", lambda: as_json('```json\n{"need": "other"}\n```') == {"need": "other"})
check("as_json garbage returns empty", lambda: as_json("not json") == {})
check("validate ok", lambda: validate_lead({"need": "pricing"}) == [])
check("validate bad enum", lambda: len(validate_lead({"need": "convert_to_paid"})) == 1)
check("validate missing need", lambda: len(validate_lead({"company": "X"})) == 1)

In [ ]:
# --- Part B: instrumentation shim ---------------------------------------
# Given: pl_init, timed, approx_token_count. You implement pl_log.
def _now_iso() -> str:
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

def pl_init(project: str = "lab12", run_group: Optional[str] = None) -> dict:
    meta = {"session_id": str(uuid.uuid4()), "project": project,
            "run_group": run_group, "created_at": _now_iso()}
    with SESSIONS_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(meta) + "\n")
    return meta

def timed(fn):
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        out = fn(*args, **kwargs)
        return out, time.perf_counter() - t0
    return wrapper

def approx_token_count(*texts: str) -> int:
    return round(sum(len(t) for t in texts) / 4)

# TODO 3. Implement pl_log.
#
# Contract: keyword-only arguments
#   run_type, name, prompt, variables, response, latency_s, approx_tokens,
#   tags=None, metadata=None, session=None
# Append exactly one JSON object per line to RUNS_PATH with these keys:
#   ts (from _now_iso), session_id and project (both from session),
#   run_type, name, prompt, variables, response,
#   latency_s (rounded to 4 places), approx_tokens, tags (default []),
#   metadata (default {}). Return None.
def pl_log(*, run_type, name, prompt, variables, response, latency_s,
           approx_tokens, tags=None, metadata=None, session=None) -> None:
    """Append one run record to runs.jsonl."""
    raise NotImplementedError

print("shim stub loaded")

## Part C - The model and the instrumented pipeline

The offline model is provided. You wire the pipeline that calls it, validates the
output, and logs each step with the right metadata.

In [ ]:
# --- Part C: deterministic offline model (given) ------------------------
# A fixed function of the prompt text. It mirrors a chat model .invoke so the
# same instrumentation code works here and against LM Studio unchanged.
class ModelResult:
    def __init__(self, content: str):
        self.content = content

def _extract_company(email: str) -> Optional[str]:
    m = re.search(r"\bat ([A-Z][\w& ]+?)[.,]", email)
    if m:
        return m.group(1).strip()
    m = re.search(r"\bfrom ([A-Z][\w& ]+?)[.,]", email)
    if m:
        return m.group(1).strip()
    return None

def _seats(email: str) -> dict[str, Optional[int]]:
    ed = re.search(r"(\d+)\s*editor", email)
    vi = re.search(r"(\d+)\s*viewer", email)
    if ed or vi:
        return {"editor": int(ed.group(1)) if ed else None,
                "viewer": int(vi.group(1)) if vi else None}
    seat = re.search(r"[Ss]eats?:?\s*(\d+)", email)
    if seat:
        return {"editor": None, "viewer": int(seat.group(1))}
    return {"editor": None, "viewer": None}

def _requirements(email: str, capture_all: bool) -> list[str]:
    reqs = []
    if "SSO" in email:
        reqs.append("SSO")
    if "SAML" in email:
        reqs.append("SAML")
    if "annual" in email.lower():
        reqs.append("annual_billing")
    if capture_all and "residency" in email.lower():
        reqs.append("data_residency")
    return reqs

def _need(email: str, baseline: bool) -> str:
    low = email.lower()
    if "residency" in low:
        return "eu_residency"
    if "pricing" in low or "price" in low:
        return "pricing"
    if "convert" in low or "trial" in low:
        return "convert_to_paid" if baseline else "trial_to_paid"
    return "other"

def _offline_extract(email: str, mode: str) -> str:
    baseline = (mode == "baseline")
    capture_all = (mode == "v3")
    obj = {
        "need": _need(email, baseline),
        "requirements": _requirements(email, capture_all),
        "seat_counts": _seats(email),
        "company": None if baseline else _extract_company(email),
    }
    return json.dumps(obj)

def _offline_summarize(lead_json_str: str) -> str:
    lead = as_json(lead_json_str)
    company = lead.get("company") or "Unknown company"
    need = lead.get("need", "other")
    reqs = lead.get("requirements", [])[:3]
    seats = lead.get("seat_counts", {}) or {}
    lines = [
        f"**Company and Need**: {company} needs {need.replace('_', ' ')}.",
        "**Requirements**: " + (", ".join(reqs) if reqs else "none stated") + ".",
    ]
    ed, vi = seats.get("editor"), seats.get("viewer")
    if ed or vi:
        parts = []
        if ed:
            parts.append(f"{ed} editor")
        if vi:
            parts.append(f"{vi} viewer")
        lines.append("**Seats**: " + " and ".join(parts) + ".")
    return "\n".join(lines)

class OfflineDeterministicModel:
    def invoke(self, messages):
        prompt = messages[-1][1] if messages else ""
        if "Executive summarizer" in prompt:
            lead_json = prompt.split("LEAD_JSON:", 1)[-1].split("OUTPUT:", 1)[0].strip()
            return ModelResult(_offline_summarize(lead_json))
        email = prompt.split("EMAIL:", 1)[-1].split("OUTPUT:", 1)[0].strip()
        if "Capture ALL stated requirements" in prompt:
            mode = "v3"
        elif "GUIDE EXAMPLES" in prompt:
            mode = "fewshot"
        else:
            mode = "baseline"
        return ModelResult(_offline_extract(email, mode))

def make_model(backend: str = BACKEND):
    if backend == "offline":
        return OfflineDeterministicModel()
    if backend in LOCAL_SERVERS:
        # LM Studio and Ollama both expose an OpenAI-compatible API, so the same
        # LangChain ChatOpenAI client points at either one. Only the base URL and
        # the model name differ. api_key is required by the client but ignored by
        # both local servers, so any non-empty string works.
        from langchain_openai import ChatOpenAI
        cfg = LOCAL_SERVERS[backend]
        return ChatOpenAI(
            model=os.getenv("LAB12_MODEL", cfg["model"]),
            temperature=0.2, timeout=45, max_retries=0,
            base_url=os.getenv("LAB12_BASE_URL", cfg["base_url"]),
            api_key=os.getenv("LAB12_API_KEY", backend),
        )
    raise ValueError(f"unknown backend {backend!r}, expected offline, lmstudio, or ollama")

_m = make_model("offline")
print("smoke:", _m.invoke([("system", "x"), ("human", "EMAIL: from Acme Inc. Need pricing. OUTPUT:")]).content)

In [ ]:
# --- Part C: pipeline ---------------------------------------------------
def dataset_hash(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()[:12]

# TODO 4. Implement render_prompt.
# Contract: replace each [[NAME]] sentinel with the matching keyword value.
#   render_prompt("EMAIL: [[EMAIL]]", EMAIL="hi {x}") returns "EMAIL: hi {x}"
#   Literal curly braces in the value must survive untouched.
def render_prompt(template: str, **fills: str) -> str:
    raise NotImplementedError

# TODO 5. Implement run_version. This is the integration step.
# Contract: for each record in the dataset,
#   1. Fill the extract prompt with the email and call the model through timed().
#   2. Parse the response with as_json and validate with validate_lead.
#   3. Log an "extract_lead" run. Its metadata must include prompt_version,
#      dataset_hash, record_id, valid (bool), and errors (list).
#   4. Only when valid, fill SUMMARIZE with the parsed JSON, call the model,
#      and log a "summarize_md" run with metadata prompt_version, dataset_hash,
#      record_id. Use the returned Markdown as the brief.
#   5. Collect one Markdown section per record into responses/briefs_<version>.md
#      Use "_(invalid extraction)_" for invalid records.
# Open a session with pl_init(project="lab12", run_group="extract_summary").
# Return the path to the written briefs file as a string.
def run_version(model, version: str, extract_prompt: str, tags: list[str]) -> str:
    raise NotImplementedError

print("pipeline stub loaded")

In [ ]:
check("render keeps literal braces",
      lambda: render_prompt("EMAIL: [[EMAIL]]", EMAIL="hi {x}") == "EMAIL: hi {x}")

### Run both prompt versions

Running v1 (baseline) and v2 (few-shot) over the three emails produces the run
log the two prompt versions will be compared from.

In [ ]:
# --- Run both prompt versions -------------------------------------------
reset_logs()
model = make_model(BACKEND)
try:
    p1 = run_version(model, "v1", EXTRACT_V1, tags=["baseline", "extract"])
    p2 = run_version(model, "v2", EXTRACT_V2, tags=["fewshot", "extract"])
    print("wrote:", p1)
    print("wrote:", p2)
except NotImplementedError:
    print("Complete TODO 4 (render_prompt) and TODO 5 (run_version), then re-run this cell.")

In [ ]:
# Confirm the run log shape
runs = [json.loads(l) for l in RUNS_PATH.read_text(encoding="utf-8").splitlines() if l.strip()] if RUNS_PATH.exists() else []
def _valid_count(ver):
    return sum(1 for r in runs
               if r["name"] == "extract_lead"
               and r["metadata"].get("prompt_version") == ver
               and r["metadata"].get("valid"))
check("11 total runs logged", lambda: len(runs) == 11)
check("v1 valid extractions equal 2", lambda: _valid_count("v1") == 2)
check("v2 valid extractions equal 3", lambda: _valid_count("v2") == 3)
check("every run carries a session_id", lambda: bool(runs) and all(r["session_id"] for r in runs))

## Part D - Filter, compare, and export

Filter the run log to a CSV, then aggregate per version into a Markdown report:
runs, token totals, valid extraction rate, and how far v2 drifted from v1.

In [ ]:
# --- Part D: filter and export ------------------------------------------
def load_runs():
    if not RUNS_PATH.exists():
        return
    for line in RUNS_PATH.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            yield json.loads(line)

def matches(rec, tags, versions) -> bool:
    ok_t = True if not tags else any(t in rec.get("tags", []) for t in tags)
    ver = (rec.get("metadata") or {}).get("prompt_version")
    ok_v = True if not versions else ver in versions
    return ok_t and ok_v

# TODO 6. Implement export_csv.
# Contract: select runs where matches(rec, tags, versions) is True, then write a
#   CSV with exactly these columns in order:
#     ts, project, run_type, name, latency_s, approx_tokens,
#     prompt_version, record_id, valid, tags
#   prompt_version, record_id, valid come from rec["metadata"].
#   tags is the run tag list joined by commas.
#   Return the number of rows written.
def export_csv(outfile: Path, tags=None, versions=None) -> int:
    raise NotImplementedError

print("export stub loaded")

In [ ]:
try:
    n = export_csv(ROOT / "exports/filtered_runs.csv", versions=["v1", "v2"])
    print("exported rows:", n)
except NotImplementedError:
    print("Complete TODO 6 (export_csv), then re-run this cell.")

In [ ]:
# --- Part D: compare versions (aggregates and output drift) -------------
def extraction_by_record(version: str) -> dict:
    out = {}
    for r in load_runs():
        meta = r.get("metadata") or {}
        if r.get("name") == "extract_lead" and meta.get("prompt_version") == version:
            out[meta.get("record_id")] = r.get("response", "")
    return out

def mean_drift(ver_a: str, ver_b: str) -> Optional[float]:
    a, b = extraction_by_record(ver_a), extraction_by_record(ver_b)
    shared = sorted(set(a) & set(b))
    if not shared:
        return None
    sims = [fuzz.ratio(a[k], b[k]) for k in shared]
    return round(100 - statistics.mean(sims), 1)

# TODO 7. Implement compare_versions.
# Contract: group runs by metadata prompt_version. For each version compute
#   runs (count), tokens_total (sum of approx_tokens),
#   valid_rate (percent of extract runs whose metadata valid is True).
# Write a Markdown report to outfile with a section per version, and return a
# dict mapping version to {"runs", "tokens_total", "valid_rate"}.
# Do not use a slash as a conjunction in the report text.
def compare_versions(outfile: Path) -> dict:
    raise NotImplementedError

print("compare stub loaded")

In [ ]:
try:
    summary = compare_versions(ROOT / "exports/version_report.md")
    print(json.dumps(summary, indent=2))
except NotImplementedError:
    summary = {}
    print("Complete TODO 7 (compare_versions), then re-run this cell.")

In [ ]:
check("v1 valid rate is 66.7", lambda: summary.get("v1", {}).get("valid_rate") == 66.7)
check("v2 valid rate is 100.0", lambda: summary.get("v2", {}).get("valid_rate") == 100.0)
check("v1 tokens_total is 596", lambda: summary.get("v1", {}).get("tokens_total") == 596)
check("v2 tokens_total is 984", lambda: summary.get("v2", {}).get("tokens_total") == 984)

## Part F - Acceptance

The acceptance cell confirms every required artifact exists. All checks green
means the lab is complete.

In [ ]:
# --- Part F: acceptance -------------------------------------------------
required = [
    ROOT / "responses/briefs_v1.md",
    ROOT / "responses/briefs_v2.md",
    RUNS_PATH,
    ROOT / "exports/filtered_runs.csv",
    ROOT / "exports/version_report.md",
]
missing = [str(p) for p in required if not p.exists()]
check("all required artifacts present", lambda: not missing)
if missing:
    print("missing:", missing)
check_summary()

## Part E - Running against a real local model (optional)

The lab ships with three backends. You have been running `offline`. To run
against a real local model, pick `lmstudio` or `ollama`. Both expose an
OpenAI-compatible server, so the same `ChatOpenAI` client drives either one. Only
the port and the model name differ.

**Option 1: LM Studio.**

1. Start LM Studio and load any chat-capable model.
2. Open the Local Server tab and start the server (default `http://localhost:1234/v1`).
3. Set `LAB12_BACKEND=lmstudio` and re-run the notebook top to bottom.

**Option 2: Ollama.**

1. Install Ollama and pull a model, for example `ollama pull llama3.1`.
2. Ollama serves an OpenAI-compatible API at `http://localhost:11434/v1` once
   it is running.
3. Set `LAB12_BACKEND=ollama` and re-run the notebook top to bottom. If your
   pulled model tag is not `llama3.1`, set `LAB12_MODEL` to the tag you pulled.

**Environment variables the backends read.**

| Variable | Purpose | Default |
|---|---|---|
| `LAB12_BACKEND` | offline, lmstudio, or ollama | offline |
| `LAB12_MODEL` | model name or tag | server-specific |
| `LAB12_BASE_URL` | override the server URL | server-specific |
| `LAB12_API_KEY` | placeholder key, ignored by local servers | backend name |

Nothing else in the notebook changes when you switch. `make_model` returns a
LangChain `ChatOpenAI` client pointed at the chosen server, and it exposes the
same `.invoke` surface the instrumentation already uses. Latency and token
numbers become real, and the valid extraction rate depends on the model you load.

**If you prefer the native Ollama client.** `langchain-ollama` provides
`ChatOllama`, which also exposes `.invoke` and drops into the pipeline unchanged.
Swap the ollama branch of `make_model` for the two lines below. The
OpenAI-compatible path is the default here because it reinforces that one client
handles every local server.

```
from langchain_ollama import ChatOllama
return ChatOllama(model=os.getenv("LAB12_MODEL", "llama3.1"), temperature=0.2,
                  base_url=os.getenv("LAB12_BASE_URL", "http://localhost:11434"))
```

To move to hosted PromptLayer later, swap the body of `pl_log` for a
`promptlayer` call. The rest of the pipeline is unaffected.

## Stretch goals

Optional. Attempt after the acceptance cell passes.

**Stretch A.** Add a third prompt `v3` that asks the extractor to capture every
stated requirement, including compliance needs. Re-run and compare against v2.
You should see v3 pick up an extra requirement that v2 missed.

**Stretch B.** Export the version comparison as a Markdown table so it can drop
straight into a report.

### Stretch A

In [ ]:
# Stretch A: write EXTRACT_V3, then
#   run_version(model, "v3", EXTRACT_V3, tags=["fewshot", "v3", "extract"])
# and re-run compare_versions. Goal: v3 captures a requirement v2 missed.
# Compare S01 under v2 and v3.

# Your code here.
print("Stretch A not attempted yet.")

### Stretch B

In [ ]:
# Stretch B: write export_table(outfile, summary) that writes a Markdown table
# with columns version, runs, tokens_total, valid_rate. Return the row count.

# Your code here.
print("Stretch B not attempted yet.")

## What to submit

- `responses/briefs_v1.md` and `responses/briefs_v2.md`
- `logging_local/artifacts/runs.jsonl`
- `exports/filtered_runs.csv` and `exports/version_report.md`
- A short `responses/notes.md` with one observation about the v1 to v2
  difference you saw in the report.

You are done when the acceptance cell prints every check as PASS.